04 样本外改为 2024-01 起
IS = 2012-08 ~ 2023-12（含 2020、2022），OOS = 2024-01 ~ 今。

注意：此前整体结果已看过 2024 年后的表现，所以这个 OOS 并非完全盲测；选参只用 IS 数据。

在 PyCharm 中选择 Python 内核，点击 **Run All / 运行全部**。最后一格调用 `main()`，无需手动逐个调用函数。


In [1]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """支持从项目根目录、pipeline 目录及项目内子目录启动。"""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file() and (candidate / '07_run_pipeline.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('找不到 pipeline 文件夹。请将 PyCharm 工作目录设为项目根目录，或设置 ETF_PROJECT_ROOT。')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """独立命名空间加载函数；不执行被加载 notebook 的入口单元格。"""
    path = PIPELINE_DIR / filename
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [2]:
def evaluate_2024(C, runs):
    ISE='2023-12-31'; OA='2024-01-01'
    best,sc=select(runs,IS_START,ISE)
    print('IS 选中（冻结）:',label(best),'| IS Sharpe',round(sc[best]['Sharpe'],3))
    rows=[]
    for name,c in [('基线',base),('选中',best)]:
        for lab,a,b in [('IS',IS_START,ISE),('OOS 2024+',OA,None)]:
            rows.append({'方案':name,'区间':lab,**fmt(metrics(runs[c][0],a=a,b=b))})
    for t in ['SPY','TQQQ']: rows.append({'方案':'买入持有 '+t,'区间':'OOS 2024+',**fmt(metrics(C[t].loc[OA:]))})
    table = pd.DataFrame(rows)
    display(table)
    return {'best': best, 'scores': sc, 'metrics': table}


In [3]:
def compare_2024(runs, sc, best):
    OA='2024-01-01'
    bm=metrics(runs[base][0],a=OA)
    oo={label(c):metrics(runs[c][0],a=OA) for c in combos if c!=base}
    isS=pd.Series({label(c):sc[c]['Sharpe'] for c in combos if c!=base}); osS=pd.Series({k:m['Sharpe'] for k,m in oo.items()})
    print('基线 OOS Sharpe',round(bm['Sharpe'],3),'CAGR',round(bm['CAGR'],3),'MDD',round(bm['MDD'],3))
    print('OOS Sharpe 超过基线:',int((osS>bm['Sharpe']).sum()),'/',len(oo),
          '| 回撤更浅:',sum(m['MDD']>bm['MDD'] for m in oo.values()),'| CAGR 更高:',sum(m['CAGR']>bm['CAGR'] for m in oo.values()))
    print('IS 超过基线 Sharpe 的组合:',int((isS>sc[base]['Sharpe']).sum()),'/',len(isS),'| IS-OOS 排名相关:',round(isS.corr(osS,method='spearman'),2))
    print('选中参数的 OOS 排名:',int(osS.rank(ascending=False)[label(best)]),'/',len(osS))
    tab=pd.DataFrame({'IS_Sharpe':isS,'OOS_Sharpe':osS,'OOS_CAGR':pd.Series({k:m['CAGR'] for k,m in oo.items()}),'OOS_MDD':pd.Series({k:m['MDD'] for k,m in oo.items()})})
    tab = tab.sort_values('OOS_Sharpe',ascending=False).round(3)
    display(tab)
    return {'ranking': tab}


In [4]:
def main(csv_path=None, context=None, runs=None):
    """本 notebook 的一键入口；返回表格、净值等结果。"""
    context = prepare_context(csv_path) if context is None else context
    O, H, L, C, tg = context
    runs = run_all(*context) if runs is None else runs
    frozen = evaluate_2024(C, runs)
    comparison = compare_2024(runs, frozen['scores'], frozen['best'])
    return {'frozen': frozen, 'comparison': comparison}


In [5]:
# 修改数据路径后，运行全部单元格。None 表示自动查找项目 data/etf_daily.csv。
CSV_PATH = None
results = main(csv_path=CSV_PATH)


数据区间: 2011-10-03 ~ 2026-10-07 | 信号分布: {'TQQQ': 2904, 'TECL': 249, 'TECS': 203, 'BSV': 150, 'UVXY': 133, 'SPXL': 17}


IS 选中（冻结）: 追踪 止损=None 止盈=0.15 | IS Sharpe 1.403


,方案,区间,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,基线,IS,0.933,0.621,1.368,2.171,-0.738,1.264
1,基线,OOS 2024+,0.875,0.614,1.331,2.053,-0.387,2.260
2,选中,IS,0.832,0.529,1.403,2.322,-0.641,1.297
3,选中,OOS 2024+,0.671,0.472,1.322,2.207,-0.372,1.802
4,买入持有 SPY,OOS 2024+,0.211,0.155,1.318,1.975,-0.188,1.127
5,买入持有 TQQQ,OOS 2024+,0.586,0.617,1.057,1.554,-0.580,1.009


基线 OOS Sharpe 1.331 CAGR 0.875 MDD -0.387
OOS Sharpe 超过基线: 18 / 35 | 回撤更浅: 34 | CAGR 更高: 2


IS 超过基线 Sharpe 的组合: 2 / 35 | IS-OOS 排名相关: 0.6
选中参数的 OOS 排名: 20 / 35


,IS_Sharpe,OOS_Sharpe,OOS_CAGR,OOS_MDD
固定 止损=0.12 止盈=0.25,1.304,1.765,0.801,-0.306
固定 止损=0.1 止盈=0.4,1.070,1.663,0.779,-0.213
固定 止损=0.06 止盈=0.25,1.158,1.626,0.527,-0.166
固定 止损=0.1 止盈=0.25,1.241,1.626,0.623,-0.211
固定 止损=0.06 止盈=0.4,0.958,1.602,0.594,-0.207
固定 止损=0.08 止盈=0.25,1.102,1.581,0.536,-0.176
固定 止损=0.08 止盈=0.4,0.921,1.570,0.604,-0.207
固定 止损=0.12 止盈=0.4,1.152,1.528,0.827,-0.327
追踪 止损=None 止盈=0.25,1.482,1.510,0.906,-0.372
固定 止损=0.1 止盈=None,1.093,1.481,0.725,-0.355
